# Chapter 7: Eigenvalues and Eigenvectors in Engineering Applications

## Eigenvalues and Eigenvectors

### Calculating Eigenvalues and Eigenvectors

#### Using SymPy for Symbolic Solutions

In [ ]:
import sympy as sp

# Define symbolic matrix
A = sp.Matrix([[2, 1, 0],
               [1, 3, 1],
               [0, 1, 2]])

# Find eigenvalues and eigenvectors
eigenvalues = A.eigenvals()
eigenvectors = A.eigenvects()

# Display results
print("Eigenvalues:")
for eigenvalue, multiplicity in eigenvalues.items():
    print(f"lambda_ = {eigenvalue}, multiplicity = {multiplicity}")

print("\nEigenvectors:")
for eigenvalue, multiplicity, vectors in eigenvectors:
    print(f"lambda_ = {eigenvalue}, multiplicity = {multiplicity}")
    for vector in vectors:
        print(f"v = {vector}")

#### Numerical Methods for Large Matrices

In [ ]:
import numpy as np

def qr_algorithm(A, max_iter=100, tol=1e-12):
    """
    Compute eigenvalues using the QR algorithm.

    Parameters:
    A : ndarray
        Input matrix
    max_iter : int
        Maximum number of iterations
    tol : float
        Convergence tolerance

    Returns:
    eigenvalues : ndarray
        Approximate eigenvalues
    """
    n = A.shape[0]
    A_k = A.copy()

    for k in range(max_iter):
        # Compute the QR decomposition
        Q, R = np.linalg.qr(A_k)

        # Form A_{k+1} = RQ
        A_next = R @ Q

        # Check for convergence
        if np.max(np.abs(np.diag(A_next) - np.diag(A_k))) < tol:
            # Extract the eigenvalues from the diagonal
            return np.diag(A_next)

        A_k = A_next

    # Extract the eigenvalues from the diagonal
    return np.diag(A_k)

# Example usage:
A = np.array([[2, 1, 0],
              [1, 3, 1],
              [0, 1, 2]])

eigenvalues = qr_algorithm(A)
print("Approximate eigenvalues:", eigenvalues)

#### Applications in Heat Transfer: Transient Heat Conduction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh

# Parameters
L = 1.0  # Length of rod (m)
nx = 20  # Number of spatial points
alpha = 0.01  # Thermal diffusivity (m**2/s)
t_final = 10.0  # Final time (s)
nt = 100  # Number of time points

# Spatial discretization
dx = L / (nx - 1)
x = np.linspace(0, L, nx)

# Construct the discretized Laplacian matrix (with insulated boundaries)
A = np.zeros((nx, nx))
for i in range(1, nx-1):
    A[i, i-1] = 1
    A[i, i] = -2
    A[i, i+1] = 1
A[0, 0] = -1
A[0, 1] = 1
A[nx-1, nx-2] = 1
A[nx-1, nx-1] = -1
A = A / dx**2

# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = eigh(A)

# Initial condition: hot in the middle, cold at the ends
T0 = np.exp(-20 * (x - L/2)**2)
T0 = T0 / np.max(T0)  # Normalize

# Express initial condition in terms of eigenvectors
coefficients = np.linalg.lstsq(eigenvectors, T0, rcond=None)[0]

# Time stepping
t = np.linspace(0, t_final, nt)
T = np.zeros((nt, nx))

for i in range(nt):
    # Compute temperature at time t[i]
    T_t = np.zeros(nx)
    for j in range(nx):
        T_t += coefficients[j] * np.exp(alpha * eigenvalues[j] * t[i]) * eigenvectors[:, j]
    T[i, :] = T_t

# Plot the results
plt.figure(figsize=(10, 6))
for i in range(0, nt, nt//5):
    plt.plot(x, T[i, :], label=f't = {t[i]:.2f} s')
plt.xlabel('Position $x$ (m)')
plt.ylabel('Temperature (normalized)')
plt.title('Transient Heat Conduction in a Rod')
plt.legend()
plt.grid(True)
plt.tight_layout()

### Power Method for Eigenvalues

#### Basic Power Method

In [ ]:
import numpy as np

def power_method(A, max_iter=100, tol=1e-10):
    """
    Compute the dominant eigenvalue and eigenvector using the power method.

    Parameters:
    A : ndarray
        Input matrix
    max_iter : int
        Maximum number of iterations
    tol : float
        Convergence tolerance

    Returns:
    lambda_max : float
        Dominant eigenvalue
    v : ndarray
        Corresponding eigenvector
    """
    n = A.shape[0]

    # Initialize with a random vector
    v = np.random.rand(n)
    v = v / np.linalg.norm(v)

    lambda_old = 0

    for i in range(max_iter):
        # Power iteration
        w = A @ v

        # Compute Rayleigh quotient (eigenvalue estimate)
        lambda_new = v.T @ A @ v

        # Normalize the vector
        v = w / np.linalg.norm(w)

        # Check for convergence
        if abs(lambda_new - lambda_old) < tol:
            return lambda_new, v

        lambda_old = lambda_new

    return lambda_old, v

#### Applications in Fluid Dynamics: Stability Analysis

In [ ]:
import numpy as np
from scipy.sparse.linalg import eigs
import matplotlib.pyplot as plt

# Example: Discretized linearized operator for channel flow
def create_channel_flow_operator(nx, ny, Re, U_max):
    """Create a simplified linearized operator for channel flow"""
    # ... (discretization details omitted for brevity)
    # This would create a large sparse matrix representing the linearized
    # Navier-Stokes operator for a given Reynolds number

    # For demonstration, we create a simpler matrix with expected properties
    n = nx * ny
    L = np.zeros((n, n))

    # Populate the matrix (this is highly simplified)
    for i in range(n):
        L[i, i] = -1.0 / Re  # Diffusion term (stabilizing)
        if i + 1 < n:
            L[i, i+1] = 0.5 * U_max / nx  # Advection term (potentially destabilizing)
        if i - 1 >= 0:
            L[i, i-1] = -0.5 * U_max / nx  # Advection term

    # Add some coupling to mimic the real operator
    for i in range(0, n-nx):
        L[i, i+nx] = 0.1 / ny
        L[i+nx, i] = 0.1 / ny

    return L

# Set up the problem
nx, ny = 20, 10
Re = 2000  # Reynolds number
U_max = 1.0  # Maximum velocity

# Create the operator
L = create_channel_flow_operator(nx, ny, Re, U_max)

# Find the eigenvalues with largest real part using a sparse eigenvalue solver
k = 5  # Number of eigenvalues to compute
eigenvalues, eigenvectors = eigs(L, k=k, which='LR')

# Stability analysis
max_real = np.max(eigenvalues.real)
if max_real > 0:
    stability = "unstable"
else:
    stability = "stable"

# Find the most unstable/least stable mode
idx_max_real = np.argmax(eigenvalues.real)
most_unstable_mode = eigenvectors[:, idx_max_real].reshape(ny, nx)

# Plot the spectrum and the mode structure side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(eigenvalues.real, eigenvalues.imag, s=50)
axes[0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[0].axvline(x=0, color='k', linestyle='--', alpha=0.3)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel('Real part (growth rate)')
axes[0].set_ylabel('Imaginary part (frequency)')
axes[0].set_title(f'Eigenvalue spectrum (Re = {Re}, flow is {stability})')

im = axes[1].contourf(np.abs(most_unstable_mode), cmap='viridis')
fig.colorbar(im, ax=axes[1], label='Mode amplitude')
axes[1].set_title(f'Most {"unstable" if max_real > 0 else "least stable"} mode structure')
axes[1].set_xlabel('x-direction (grid index)')
axes[1].set_ylabel('y-direction (grid index)')
plt.tight_layout()

## Stability Analysis of Fluid Flows Using Eigenvalue Decomposition

### Numerical Implementation with Python

In [ ]:
def create_channel_flow_operator(nx, ny, Re, U_max):
    """Create a simplified linearized operator for channel flow"""
    n = nx * ny
    L = np.zeros((n, n))

    # Populate the matrix with physics terms
    for i in range(n):
        # Diffusion term (stabilizing)
        L[i, i] = -1.0 / Re

        # Advection terms (potentially destabilizing)
        if i + 1 < n:
            L[i, i+1] = 0.5 * U_max / nx
        if i - 1 >= 0:
            L[i, i-1] = -0.5 * U_max / nx

    # Spatial coupling terms
    for i in range(0, n-nx):
        L[i, i+nx] = 0.1 / ny
        L[i+nx, i] = 0.1 / ny

    return L

In [ ]:
# Find eigenvalues with largest real part
eigenvalues, eigenvectors = eigs(L, k=5, which='LR')

# Stability analysis
max_real = np.max(eigenvalues.real)
if max_real > 0:
    stability = "unstable"
else:
    stability = "stable"

### Symbolic Analysis with SymPy

In [ ]:
import sympy as sp

# Create symbolic variables
Re = sp.Symbol('Re', positive=True)
U = sp.Symbol('U', positive=True)

# Create a simplified symbolic linearized operator
L = sp.Matrix([
    [-1/Re,    U/2,    0.1],
    [-U/2,   -1/Re,    0.1],
    [ 0.1,     0.1,  -1/Re]
])

# Calculate eigenvalues symbolically
eigenvalues = L.eigenvals()

### Physical Interpretation of Eigenmodes

In [ ]:
# Find the most unstable/least stable mode
idx_max_real = np.argmax(eigenvalues.real)
most_unstable_mode = eigenvectors[:, idx_max_real].reshape(ny, nx)

# Visualize the mode structure
plt.contourf(np.abs(most_unstable_mode), cmap='viridis')

### Problems and Exercises

#### Applications in Heat Transfer: Modal Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def power_method(A, max_iter=100, tol=1e-10):
    """
    Compute the dominant eigenvalue and eigenvector using the power method.

    Parameters:
    A : ndarray
        Input matrix
    max_iter : int
        Maximum number of iterations
    tol : float
        Convergence tolerance

    Returns:
    lambda_max : float
        Dominant eigenvalue
    v : ndarray
        Corresponding eigenvector
    convergence_history : list
        History of eigenvalue estimates
    """
    n = A.shape[0]

    # Initialize with a random vector
    np.random.seed(42)  # For reproducibility
    v = np.random.rand(n)
    v = v / np.linalg.norm(v)

    lambda_old = 0
    convergence_history = []

    for i in range(max_iter):
        # Power iteration
        w = A @ v

        # Compute Rayleigh quotient (eigenvalue estimate)
        lambda_new = v.T @ A @ v

        # Track convergence
        convergence_history.append(lambda_new)

        # Normalize the vector
        v = w / np.linalg.norm(w)

        # Check for convergence
        if abs(lambda_new - lambda_old) < tol:
            return lambda_new, v, convergence_history

        lambda_old = lambda_new

    return lambda_old, v, convergence_history

def create_2d_heat_transfer_matrix(nx, ny, kx, ky, dx, dy):
    """Create a 2D heat transfer matrix for rectangular domain"""
    n = nx * ny
    A = np.zeros((n, n))

    for i in range(ny):
        for j in range(nx):
            idx = i * nx + j

            # Diagonal term
            A[idx, idx] = -2 * kx / dx**2 - 2 * ky / dy**2

            # x-direction neighbors
            if j > 0:  # Left neighbor
                A[idx, idx-1] = kx / dx**2
            if j < nx-1:  # Right neighbor
                A[idx, idx+1] = kx / dx**2

            # y-direction neighbors
            if i > 0:  # Bottom neighbor
                A[idx, idx-nx] = ky / dy**2
            if i < ny-1:  # Top neighbor
                A[idx, idx+nx] = ky / dy**2

    return A

# Set up the problem
nx, ny = 20, 20
dx, dy = 0.05, 0.05
kx, ky = 1.0, 0.5  # Thermal conductivities in x and y directions

# Create the heat transfer matrix
A = create_2d_heat_transfer_matrix(nx, ny, kx, ky, dx, dy)

# Use the power method to find the dominant mode
lambda_max, v_max, convergence_history = power_method(A, max_iter=1000)

# Reshape the eigenvector to 2D for visualization
mode_shape = v_max.reshape(ny, nx)

# Create a figure with subplots
fig = plt.figure(figsize=(15, 5))

# Plot 1: Dominant thermal mode
ax1 = fig.add_subplot(131)
im1 = ax1.contourf(mode_shape, cmap='hot', levels=20)
plt.colorbar(im1, ax=ax1, label='Mode amplitude (normalized)')
ax1.set_title(f'Largest-magnitude thermal mode\n(lambda = {lambda_max:.1f} 1/s)')
ax1.set_xlabel('x-index')
ax1.set_ylabel('y-index')

# Plot 2: Thermal conductivity tensor visualization
ax2 = fig.add_subplot(132)
theta = np.linspace(0, 2*np.pi, 100)
x = np.sqrt(kx) * np.cos(theta)
y = np.sqrt(ky) * np.sin(theta)
ax2.plot(x, y, 'b-', linewidth=2)
ax2.add_patch(plt.Circle((0, 0), radius=0.05, color='r'))
ax2.set_xlim(-1.5, 1.5)
ax2.set_ylim(-1.5, 1.5)
ax2.set_aspect('equal')
ax2.grid(True)
ax2.set_title('Thermal Conductivity Tensor')
ax2.set_xlabel('x-direction (kx = 1.0 W/(m K))')
ax2.set_ylabel('y-direction (ky = 0.5 W/(m K))')

# Plot 3: Convergence of power method
ax3 = fig.add_subplot(133)
iterations = np.arange(len(convergence_history))
ax3.plot(iterations, convergence_history, 'g-')
ax3.axhline(y=lambda_max, color='r', linestyle='--', label=f'Final lambda = {lambda_max:.1f}')
ax3.set_title('Power Method Convergence')
ax3.set_xlabel('Iteration')
ax3.set_ylabel('Eigenvalue estimate (1/s)')
ax3.set_xlim(0, min(50, len(convergence_history)))  # Focus on first 50 iterations
ax3.legend()
ax3.grid(True)

plt.tight_layout()
plt.show()

## Advanced Power Method and QR Algorithm

### Power Method Enhancements

#### Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def inverse_power_method(A, x0, mu=0, max_iterations=1000, tolerance=1e-10):
    """
    Find the eigenvalue of A closest to the shift mu by iterating
    with (A - mu*I)^{-1}; mu = 0 gives the smallest-magnitude eigenvalue.
    """
    n = len(x0)
    x = x0.copy()
    eigenvalues = []

    A_shifted = A - mu * np.eye(n)

    for k in range(max_iterations):
        # Solve (A - mu*I) * y = x for y
        try:
            y = np.linalg.solve(A_shifted, x)
        except np.linalg.LinAlgError:
            print("Matrix is singular or ill-conditioned")
            break

        lambda_k = 1.0 / (np.dot(x, y) / np.dot(x, x))
        lambda_actual = lambda_k + mu  # Actual eigenvalue

        # Normalize
        x_new = y / np.linalg.norm(y)

        # Check convergence
        error = np.linalg.norm(x_new - x)
        eigenvalues.append(lambda_actual)

        if error < tolerance:
            print(f"Inverse power: Converged in {k+1} iterations")
            return lambda_actual, x_new, eigenvalues

        x = x_new

    return eigenvalues[-1], x, eigenvalues

# Test matrix
A = np.array([[5., 2., 1.],
              [2., 6., 2.],
              [1., 2., 4.]])

x0 = np.ones(3)

# Basic power method (largest eigenvalue)
print("All eigenvalues of A:")
eigs = np.linalg.eigvalsh(A)
print(sorted(eigs, reverse=True))

# Inverse power method (smallest eigenvalue)
lambda_min, v_min, conv_inv = inverse_power_method(A, x0)
print(f"\nSmallest eigenvalue: {lambda_min:.6f}")
print(f"Eigenvector: {v_min}")

# Shifted power method (eigenvalue near 5.5)
lambda_shifted, v_shifted, conv_shift = inverse_power_method(A, x0, mu=5.5)
print(f"\nEigenvalue nearest 5.5: {lambda_shifted:.6f}")
print(f"Eigenvector: {v_shifted}")

# Convergence comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Direct power method for largest eigenvalue
def power_method_track(A, x0, max_iterations=100):
    x = x0.copy()
    eigenvalues = []
    for k in range(max_iterations):
        y = A @ x
        lambda_k = np.dot(x, y) / np.dot(x, x)
        x_new = y / np.linalg.norm(y)
        eigenvalues.append(lambda_k)
        x = x_new
    return eigenvalues

eigs_power = power_method_track(A, x0)

axes[0].semilogy(np.abs(np.array(eigs_power) - eigs[-1]), 'b-', linewidth=2, label='Power method')
axes[0].semilogy(np.abs(np.array(conv_inv) - eigs[0]), 'r-', linewidth=2, label='Inverse power')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('|Eigenvalue error|')
axes[0].set_title('Convergence Comparison')
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(True)

# Eigenvalue location plot
axes[1].scatter(eigs, np.zeros_like(eigs), s=100, c='blue', marker='o', label='Eigenvalues')
axes[1].scatter([lambda_min], [0], s=200, c='red', marker='*', label='Inverse power result')
axes[1].scatter([lambda_shifted], [0], s=200, c='green', marker='^', label='Shifted power result')
axes[1].set_xlabel('Eigenvalue')
axes[1].set_ylabel('')
axes[1].set_title('Eigenvalue Spectrum')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_yticks([])

plt.tight_layout()
plt.show()

### QR Algorithm

#### QR Algorithm with Shifts

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def qr_algorithm(A, max_iterations=1000, tolerance=1e-10):
    """
    Compute eigenvalues using the QR algorithm.
    """
    A_k = A.copy()
    n = A.shape[0]
    iterations_list = []

    for k in range(max_iterations):
        # QR decomposition
        Q, R = np.linalg.qr(A_k)

        # Form A_{k+1} = R*Q
        A_k_new = R @ Q

        # Check convergence (subdiagonal elements small)
        off_diag = np.sum(np.abs(np.tril(A_k_new, -1)))
        iterations_list.append(off_diag)

        if off_diag < tolerance:
            print(f"QR algorithm converged in {k+1} iterations")
            # Extract eigenvalues from diagonal
            eigenvalues = np.diag(A_k_new)
            return np.sort(eigenvalues), A_k_new, iterations_list

        A_k = A_k_new

    print(f"QR algorithm did not converge after {max_iterations} iterations")
    eigenvalues = np.diag(A_k)
    return np.sort(eigenvalues), A_k, iterations_list

def qr_algorithm_shifted(A, max_iterations=1000, tolerance=1e-10):
    """
    QR algorithm with Rayleigh quotient shifts (more efficient).
    """
    A_k = A.copy()
    n = A.shape[0]
    iterations_list = []

    for k in range(max_iterations):
        # Rayleigh quotient shift (use bottom-right element)
        mu_k = A_k[-1, -1]

        # QR decomposition of shifted matrix
        Q, R = np.linalg.qr(A_k - mu_k * np.eye(n))

        # Form A_{k+1}
        A_k_new = R @ Q + mu_k * np.eye(n)

        # Check convergence (subdiagonal elements small)
        off_diag = np.sum(np.abs(np.tril(A_k_new, -1)))
        iterations_list.append(off_diag)

        if off_diag < tolerance:
            print(f"QR algorithm (shifted) converged in {k+1} iterations")
            eigenvalues = np.diag(A_k_new)
            return np.sort(eigenvalues), A_k_new, iterations_list

        A_k = A_k_new

    print(f"QR algorithm (shifted) did not converge after {max_iterations} iterations")
    eigenvalues = np.diag(A_k)
    return np.sort(eigenvalues), A_k, iterations_list

# Test matrix
A = np.array([[1., 2., 3., 4.],
              [2., 5., 6., 7.],
              [3., 6., 8., 9.],
              [4., 7., 9., 10.]])

# Compute eigenvalues using QR algorithms
eigs_qr, A_qr, iter_qr = qr_algorithm(A, max_iterations=500)
eigs_qr_shifted, A_qr_shifted, iter_qr_shifted = qr_algorithm_shifted(A, max_iterations=500)

# Reference: numpy's eigenvalue solver
eigs_numpy = np.sort(np.linalg.eigvalsh(A))

print("\nEigenvalues from QR algorithm:")
print(eigs_qr)
print("\nEigenvalues from QR algorithm (shifted):")
print(eigs_qr_shifted)
print("\nReference (numpy):")
print(eigs_numpy)

# Plot convergence
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].semilogy(iter_qr, 'b-', linewidth=2, label='QR algorithm')
axes[0].semilogy(iter_qr_shifted, 'r-', linewidth=2, label='QR algorithm (shifted)')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Sum of subdiagonal magnitudes')
axes[0].set_title('QR Algorithm Convergence')
axes[0].legend()
axes[0].grid(True)

# Final triangular matrices
im = axes[1].imshow(np.triu(A_qr_shifted), cmap='RdBu_r', aspect='auto')
axes[1].set_title('Upper Triangular Form (shifted QR)')
axes[1].set_xlabel('Column index')
axes[1].set_ylabel('Row index')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

## Schur Decomposition

#### Python Implementation

In [ ]:
import numpy as np
from scipy.linalg import schur

# Test with a non-symmetric matrix
A = np.array([[1., 2., 3.],
              [0., 4., 5.],
              [0., 0., 6.]])

print("Original matrix A:")
print(A)

# Compute Schur decomposition
T, Z = schur(A)

print("\nSchur form T (upper triangular):")
print(T)
print("\nUnitary matrix Z:")
print(Z)

# Verify: A = Z * T * Z^H
A_reconstructed = Z @ T @ Z.conj().T
print("\nReconstruction error ||A - Z*T*Z^H||:")
print(np.linalg.norm(A - A_reconstructed))

# Eigenvalues are the diagonal of T
print("\nEigenvalues from Schur form (diagonal of T):")
print(np.diag(T))

# Verify orthonormality
print("\nOrthonormality check Z^H*Z:")
print(Z.conj().T @ Z)

# For a non-symmetric matrix
A_nonsym = np.array([[1., 2., 3.],
                     [4., 5., 6.],
                     [7., 8., 9.]])

T_nonsym, Z_nonsym = schur(A_nonsym)

print("\n\nNon-symmetric matrix:")
print("Eigenvalues from Schur form:")
print(np.diag(T_nonsym))
print("\nEigenvalues from eig:")
eigs = np.linalg.eigvals(A_nonsym)
print(eigs)

## Applications: PCA, Markov Chains, and PageRank

### Principal Component Analysis (PCA)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generate synthetic data with correlation
np.random.seed(42)
n_samples = 200
theta = np.pi / 6  # 30 degree rotation

# Base data
u = np.random.randn(n_samples, 2)
u[:, 1] *= 0.3  # Different variances

# Rotate
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta), np.cos(theta)]])
X = u @ R.T

# Center data
X_centered = X - X.mean(axis=0)

# Covariance matrix
C = (X_centered.T @ X_centered) / (n_samples - 1)

print("Covariance matrix:")
print(C)

# Eigenvalue decomposition
eigenvalues, eigenvectors = np.linalg.eigh(C)

# Sort by eigenvalues
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("\nEigenvalues:")
print(eigenvalues)
print("\nEigenvectors (principal components):")
print(eigenvectors)

# Explained variance ratio
explained_variance_ratio = eigenvalues / eigenvalues.sum()
print("\nExplained variance ratio:")
print(explained_variance_ratio)

# Project onto principal components
X_pca = X_centered @ eigenvectors

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original data with eigenvectors
axes[0].scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.5, s=30)
for i in range(2):
    scale = 3 * np.sqrt(eigenvalues[i])
    axes[0].arrow(0, 0, scale*eigenvectors[0, i], scale*eigenvectors[1, i],
                  head_width=0.2, head_length=0.1, fc=f'C{i}', ec=f'C{i}', linewidth=2)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].set_title('Original Data with Principal Components')
axes[0].grid(True)
axes[0].axis('equal')

# PCA space
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, s=30)
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title('Data in PCA Space')
axes[1].grid(True)
axes[1].axis('equal')

plt.tight_layout()
plt.show()

# Dimension reduction: use only first principal component
X_reduced = X_pca[:, :1]
print(f"\nDimension reduction: {X_centered.shape[1]} -> {X_reduced.shape[1]}")
print(f"Variance retained: {explained_variance_ratio[0]:.2%}")

### Markov Chains and Steady-State Distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Transition probability matrix for a 4-state Markov chain
# P[i,j] = probability of transition from state i to state j
P = np.array([[0.7, 0.2, 0.1, 0.0],
              [0.0, 0.6, 0.3, 0.1],
              [0.1, 0.1, 0.7, 0.1],
              [0.2, 0.1, 0.0, 0.7]])

print("Transition probability matrix P:")
print(P)

# Check that rows sum to 1 (stochastic matrix)
print("\nRow sums (should be 1):")
print(P.sum(axis=1))

# Eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(P.T)

print("\nEigenvalues of P^T:")
print(eigenvalues)

# Find eigenvector for eigenvalue 1
idx_one = np.argmax(np.abs(eigenvalues - 1.0) < 1e-10)
steady_state = eigenvectors[:, idx_one].real
steady_state /= steady_state.sum()  # Normalize

print("\nSteady-state distribution:")
print(steady_state)

# Verify by iterating P
x = np.array([1., 0., 0., 0.])  # Start in state 0
iterations = []
for k in range(100):
    x = P.T @ x
    iterations.append(x.copy())

iterations = np.array(iterations)

# Plot convergence to steady state
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Convergence of each state
for i in range(4):
    axes[0].plot(iterations[:, i], label=f'State {i}', linewidth=2)
    axes[0].axhline(y=steady_state[i], color=f'C{i}', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Probability')
axes[0].set_title('Convergence to Steady-State')
axes[0].legend()
axes[0].grid(True)

# Steady-state distribution
axes[1].bar(range(4), steady_state, color='skyblue', edgecolor='navy')
axes[1].set_xlabel('State')
axes[1].set_ylabel('Probability')
axes[1].set_title('Steady-State Distribution')
axes[1].set_ylim([0, 0.5])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Mean recurrence times
print("\nMean recurrence times:")
for i in range(4):
    mrt = 1 / steady_state[i]
    print(f"State {i}: {mrt:.2f}")

### Google PageRank Algorithm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simplified web graph
# Page 1 links to pages 2 and 3
# Page 2 links to page 1
# Page 3 links to pages 1 and 2
# Page 4 links to page 3

# Adjacency matrix (outgoing links)
links = np.array([[0, 1, 1, 0],    # Page 1 -> 2, 3
                  [1, 0, 0, 0],    # Page 2 -> 1
                  [1, 1, 0, 0],    # Page 3 -> 1, 2
                  [0, 0, 1, 0]])   # Page 4 -> 3

print("Link structure:")
print(links)

# Out-degree for each page
out_degree = links.sum(axis=1)
print("\nOut-degree:")
print(out_degree)

# Transition matrix (with dangling node handling)
# P[i,j] = probability of following a link from page i to page j
P = np.zeros_like(links, dtype=float)
for i in range(4):
    if out_degree[i] > 0:
        P[i, :] = links[i, :] / out_degree[i]
    else:
        P[i, :] = 1 / 4  # Dangling node: link to all pages equally

print("\nTransition matrix P:")
print(P)

# PageRank with damping factor
damping_factor = 0.85
n_pages = 4

# Google matrix: G = (1-d)/n * 1*1^T + d*P^T
# where d is damping factor
G = ((1 - damping_factor) / n_pages) * np.ones((n_pages, n_pages)) + damping_factor * P.T

print("\nGoogle matrix G:")
print(G)

# PageRank is the dominant eigenvector of G
eigenvalues, eigenvectors = np.linalg.eig(G)

# Find dominant eigenvector (eigenvalue ~1)
idx_max = np.argmax(np.real(eigenvalues))
pagerank = np.real(eigenvectors[:, idx_max])
pagerank /= pagerank.sum()  # Normalize

print("\nPageRank scores:")
for i in range(n_pages):
    print(f"Page {i+1}: {pagerank[i]:.4f}")

# Verify by power iteration
pr = np.ones(n_pages) / n_pages
for k in range(100):
    pr = G @ pr

print("\nPageRank (power iteration):")
for i in range(n_pages):
    print(f"Page {i+1}: {pr[i]:.4f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart of PageRank
axes[0].bar(range(1, n_pages+1), pagerank, color='steelblue', edgecolor='navy')
axes[0].set_xlabel('Page')
axes[0].set_ylabel('PageRank')
axes[0].set_title('PageRank Scores')
axes[0].set_xticks(range(1, n_pages+1))
axes[0].grid(True, alpha=0.3, axis='y')

# Network visualization (simplified)
# Plot pages and links
positions = np.array([[0, 2], [2, 2], [2, 0], [4, 0]])
axes[1].scatter(positions[:, 0], positions[:, 1], s=500*pagerank/pagerank.max(),
                c=pagerank, cmap='YlOrRd', edgecolors='black', linewidth=2, zorder=3)

# Draw links
for i in range(n_pages):
    for j in range(n_pages):
        if links[i, j] > 0:
            axes[1].annotate('', xy=positions[j], xytext=positions[i],
                            arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))

# Label pages
for i in range(n_pages):
    axes[1].text(positions[i, 0], positions[i, 1], f'P{i+1}',
                ha='center', va='center', fontsize=12, fontweight='bold')

axes[1].set_xlim(-1, 5)
axes[1].set_ylim(-1, 3)
axes[1].set_aspect('equal')
axes[1].set_title('Web Graph (bubble size = PageRank)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Generalized Eigenvalue Problems

### Theory and Applications

#### Application: Vibration Analysis

In [ ]:
import numpy as np
from scipy.linalg import eig
import matplotlib.pyplot as plt

# Two-degree-of-freedom spring-mass system
# m1 = m2 = 1 kg
# k1 = 2 N/m, k12 = 1 N/m, k2 = 2 N/m
#
# Equations:
# m1*x1'' = -k1*x1 + k12*(x2-x1)
# m2*x2'' = -(k12)*(x2-x1) - k2*x2

# Mass matrix
M = np.array([[1., 0.],
              [0., 1.]])

# Stiffness matrix
K = np.array([[2. + 1., -1.],
              [-1., 1. + 2.]])

print("Mass matrix M:")
print(M)
print("\nStiffness matrix K:")
print(K)

# Solve generalized eigenvalue problem: K*v = omega^2 * M*v
eigenvalues, eigenvectors = eig(K, M)

# eig returns eigenvalues in no particular order; sort ascending
idx = np.argsort(eigenvalues.real)
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Natural frequencies (omega = sqrt(lambda))
omega_squared = eigenvalues.real
omega = np.sqrt(omega_squared)

print("\nNatural frequencies (rad/s):")
for i in range(len(omega)):
    print(f"omega_{i+1} = {omega[i]:.4f} rad/s")

print("\nMode shapes:")
for i in range(len(omega)):
    print(f"Mode {i+1}: {eigenvectors[:, i].real}")

# Verify solution
print("\nVerification (should be near zero):")
for i in range(len(omega)):
    error = np.linalg.norm(K @ eigenvectors[:, i].real - omega_squared[i] * M @ eigenvectors[:, i].real)
    print(f"Error in mode {i+1}: {error:.2e}")

# Visualize mode shapes
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Mode 1
mode1 = eigenvectors[:, 0].real
mode1 /= np.max(np.abs(mode1))
axes[0].bar([1, 2], mode1, color='steelblue', edgecolor='navy', width=0.4)
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].set_ylabel('Displacement (normalized)')
axes[0].set_title(rf'Mode 1: $\omega_1$ = {omega[0]:.3f} rad/s')
axes[0].set_xticks([1, 2])
axes[0].set_xticklabels(['m1', 'm2'])
axes[0].set_ylim([-1.5, 1.5])
axes[0].grid(True, alpha=0.3)

# Mode 2
mode2 = eigenvectors[:, 1].real
mode2 /= np.max(np.abs(mode2))
axes[1].bar([1, 2], mode2, color='coral', edgecolor='darkred', width=0.4)
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].set_ylabel('Displacement (normalized)')
axes[1].set_title(rf'Mode 2: $\omega_2$ = {omega[1]:.3f} rad/s')
axes[1].set_xticks([1, 2])
axes[1].set_xticklabels(['m1', 'm2'])
axes[1].set_ylim([-1.5, 1.5])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Spectral Graph Theory Basics

### Graph Laplacian and Spectral Properties

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

# Two triangles joined by a single bridge edge (2, 3):
# a graph with two natural communities
edges = [(0, 1), (1, 2), (2, 0), (3, 4), (4, 5), (5, 3), (2, 3)]
n_nodes = 6

# Adjacency matrix
A = np.zeros((n_nodes, n_nodes))
for i, j in edges:
    A[i, j] = 1
    A[j, i] = 1

print("Adjacency matrix A:")
print(A)

# Degree matrix
degrees = A.sum(axis=1)
D = np.diag(degrees)

print("\nDegree of each node:", degrees)

# Laplacian
L = D - A

print("\nLaplacian matrix L:")
print(L)

# Eigenvalues and eigenvectors of L
eigenvalues_L, eigenvectors_L = np.linalg.eigh(L)

print("\nEigenvalues of Laplacian:")
print(eigenvalues_L)

# Spectral properties
print(f"\nNumber of connected components: {np.sum(eigenvalues_L < 1e-10)}")
print(f"Algebraic connectivity (lambda__2): {eigenvalues_L[1]:.4f}")

# Fiedler vector (eigenvector of lambda__2)
fiedler = eigenvectors_L[:, 1]
print("\nFiedler vector:", fiedler)

# Spectral clustering (divide nodes by Fiedler vector sign)
clusters = (fiedler > 0).astype(int)
print("Clusters:", clusters)

# Create NetworkX graph for visualization
G = nx.Graph()
G.add_edges_from(edges)

# Draw the graph
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original graph with node colors by cluster
pos = nx.spring_layout(G, seed=42)
colors = ['red' if clusters[i] == 0 else 'blue' for i in range(n_nodes)]
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=500, ax=axes[0])
nx.draw_networkx_edges(G, pos, ax=axes[0])
nx.draw_networkx_labels(G, pos, ax=axes[0])
axes[0].set_title('Graph with Spectral Clustering')
axes[0].axis('off')

# Eigenvalue spectrum
axes[1].stem(range(len(eigenvalues_L)), eigenvalues_L, basefmt=' ')
axes[1].set_xlabel('Eigenvalue index')
axes[1].set_ylabel('Eigenvalue')
axes[1].set_title('Laplacian Eigenvalue Spectrum')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute Fiedler cut (number of edges between the two clusters)
print("\nFiedler cut analysis:")
print(f"Cut between clusters: {int(np.sum(A[clusters == 0, :][:, clusters == 1]))}")

## Matrix Decompositions: Cholesky, LDL, Polar

### Cholesky Decomposition

In [ ]:
import numpy as np
from scipy.linalg import cholesky, solve_triangular
import matplotlib.pyplot as plt

# Create positive-definite matrix
A = np.array([[4, 2, 1],
              [2, 5, 2],
              [1, 2, 3]], dtype=float)

# Compute Cholesky decomposition
L = cholesky(A, lower=True)
print("Lower triangular L:")
print(L)

# Verify: A = L @ L.T
print(f"Reconstruction error: {np.linalg.norm(A - L @ L.T):.2e}")

# Solve Ax = b using Cholesky
b = np.array([7, 8, 6], dtype=float)

# Forward substitution: Ly = b
y = solve_triangular(L, b, lower=True)

# Backward substitution: L.T x = y
x = solve_triangular(L.T, y, lower=False)

print(f"Solution: {x}")
print(f"Verification: {A @ x}")
print(f"Should equal: {b}")

# Compute determinant via Cholesky
det_L = np.prod(np.diag(L))
det_A = det_L ** 2
print(f"Determinant via Cholesky: {det_A:.6f}")
print(f"Determinant direct: {np.linalg.det(A):.6f}")

# Cholesky gives a "factor" L with L @ L.T = A, but the matrix
# square root sqrt(A) is a *different* object: it must satisfy
# sqrt(A) @ sqrt(A) = A. For symmetric positive-definite A, the
# principal square root is computed by scipy.linalg.sqrtm.
from scipy.linalg import sqrtm
sqrt_A = sqrtm(A)
print(f"Matrix square root sqrt(A):")
print(sqrt_A)
print(f"Verification (sqrt(A) @ sqrt(A) = A): "
      f"{np.linalg.norm(sqrt_A @ sqrt_A - A):.2e}")
print(f"For comparison, ||L @ L - A|| (NOT zero in general): "
      f"{np.linalg.norm(L @ L - A):.2e}")
print(f"  whereas the Cholesky identity is ||L @ L.T - A|| = "
      f"{np.linalg.norm(L @ L.T - A):.2e}")

### LDL Decomposition

In [ ]:
import numpy as np
from scipy.linalg import ldl

# Symmetric matrix (not necessarily positive-definite)
A = np.array([[4, 2, 1],
              [2, 3, -1],
              [1, -1, 2]], dtype=float)

# LDL decomposition
L, D, perm = ldl(A)

print("L (unit lower triangular):")
print(L)
print("\nD (diagonal):")
print(np.diag(D))

# Permutation matrix P
P = np.eye(3)[perm]
print("\nPermutation matrix P:")
print(P.astype(int))

# Verify: P.T @ A @ P = L @ D @ L.T
PAP = P.T @ A @ P
LDL_T = L @ D @ L.T

print(f"Reconstruction error: {np.linalg.norm(PAP - LDL_T):.2e}")

# Solve system using LDL
b = np.array([3, 2, 1], dtype=float)
b_perm = b[perm]

# Forward substitution: L y = b_perm
y = np.linalg.solve(L, b_perm)

# Diagonal scaling: D z = y
z = np.linalg.solve(D, y)

# Backward substitution: L.T x_perm = z
x_perm = np.linalg.solve(L.T, z)

# Unpermute
x = x_perm[np.argsort(perm)]

print(f"Solution: {x}")
print(f"Verification: {A @ x}")
print(f"Should equal: {b}")

### Polar Decomposition

In [ ]:
import numpy as np
from scipy.linalg import svd, sqrtm

# Matrix for polar decomposition
A = np.array([[3, 1],
              [1, 2]], dtype=float)

# Method 1: Via SVD
# A = U_svd @ Sigma @ Vt
# Polar: A = U @ P where U = U_svd @ Vt and P = V @ Sigma @ Vt
U_svd, sigma, Vt = svd(A)
U_polar = U_svd @ Vt
P_polar = Vt.T @ np.diag(sigma) @ Vt

print("Polar decomposition A = U @ P:")
print(f"U (unitary):\n{U_polar}")
print(f"P (symmetric positive-semidefinite):\n{P_polar}")

# Verify: A = U @ P
print(f"Reconstruction error: {np.linalg.norm(A - U_polar @ P_polar):.2e}")

# Verify U is orthogonal
print(f"U^T @ U = I: {np.linalg.norm(U_polar.T @ U_polar - np.eye(2)):.2e}")

# Verify P is symmetric and positive-semidefinite
print(f"P is symmetric: {np.linalg.norm(P_polar - P_polar.T):.2e}")
print(f"P eigenvalues (should be >= 0): {np.linalg.eigvals(P_polar)}")

# Compute matrix square root via polar
# P = sqrt(A^T A), so sqrt(A) can be approximated
AtA = A.T @ A
P_sqrtAtA = sqrtm(AtA)
print(f"sqrt(A^T A):\n{P_sqrtAtA}")

# Geometric interpretation: A represents rotation + scaling
# U is pure rotation, P is pure scaling
print(f"\nSingular values (scaling factors): {sigma}")

## Dynamical Systems via Eigenvalue Analysis

### Stability of Discrete Dynamical Systems

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Example 1: Stable system
A_stable = np.array([[0.8, 0.1], [0, 0.7]])
evals_stable = np.linalg.eigvals(A_stable)
print(f"Stable system eigenvalues: {evals_stable}")
print(f"Magnitudes: {np.abs(evals_stable)}")

# Example 2: Unstable system
A_unstable = np.array([[1.2, 0.1], [0, 0.9]])
evals_unstable = np.linalg.eigvals(A_unstable)
print(f"\nUnstable system eigenvalues: {evals_unstable}")
print(f"Magnitudes: {np.abs(evals_unstable)}")

# Simulate trajectories
def simulate_system(A, x0, n_steps):
    x = x0.copy()
    trajectory = [x.copy()]
    for _ in range(n_steps):
        x = A @ x
        trajectory.append(x.copy())
    return np.array(trajectory)

# Initial conditions on unit circle
angles = np.linspace(0, 2*np.pi, 8, endpoint=False)
x0_list = np.array([[np.cos(a), np.sin(a)] for a in angles])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Stable system
for x0 in x0_list:
    traj = simulate_system(A_stable, x0, 20)
    axes[0].plot(traj[:, 0], traj[:, 1], 'b-', alpha=0.5)
    axes[0].plot(x0[0], x0[1], 'bo', markersize=4)

axes[0].set_xlim(-1.5, 1.5)
axes[0].set_ylim(-1.5, 1.5)
axes[0].set_aspect('equal')
axes[0].set_title('Stable System (all |eigenvalues| < 1)')
axes[0].set_xlabel('$x_1$')
axes[0].set_ylabel('$x_2$')
axes[0].grid(True)

# Unstable system
for x0 in x0_list:
    traj = simulate_system(A_unstable, x0, 10)
    axes[1].plot(traj[:, 0], traj[:, 1], 'r-', alpha=0.5)
    axes[1].plot(x0[0], x0[1], 'ro', markersize=4)

axes[1].set_xlim(-3, 3)
axes[1].set_ylim(-2, 2)
axes[1].set_aspect('equal')
axes[1].set_title('Unstable System (one |eigenvalue| > 1)')
axes[1].set_xlabel('$x_1$')
axes[1].set_ylabel('$x_2$')
axes[1].grid(True)

plt.tight_layout()
plt.show()

### Leslie Matrices for Population Dynamics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Leslie matrix for human population (simplified)
# Age classes: 0-15, 15-30, 30-45, 45-60, 60+
fertility = np.array([0, 0.8, 1.0, 0.3, 0])  # births per female
survival = np.array([0.98, 0.95, 0.92, 0.80])  # to next age class

# Construct Leslie matrix
n_classes = len(fertility)
L = np.zeros((n_classes, n_classes))
L[0, :] = fertility
for i in range(n_classes - 1):
    L[i+1, i] = survival[i]

print("Leslie Matrix:")
print(L)

# Find growth rate: the Perron root is the eigenvalue of largest modulus
evals, evecs = np.linalg.eig(L)
idx_dominant = np.argmax(np.abs(evals))
lambda1 = evals[idx_dominant].real
stable_dist = evecs[:, idx_dominant].real
stable_dist = stable_dist / stable_dist.sum()

print(f"Dominant eigenvalue (growth rate): {lambda1:.6f}")
print(f"Stable age distribution: {stable_dist}")

# Simulate population
initial_pop = np.array([1000, 800, 600, 400, 200])
n_years = 50
population = [initial_pop.copy()]

x = initial_pop.copy()
for year in range(n_years):
    x = L @ x
    population.append(x.copy())

population = np.array(population)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Total population vs. pure exponential growth at rate lambda_1
total_pop = population.sum(axis=1)
years = np.arange(n_years + 1)
axes[0].semilogy(years, total_pop, 'b-', linewidth=2, label='Simulated total')
axes[0].semilogy(years, total_pop[0] * lambda1**years,
                 'r--', linewidth=1.5, label='lambda_1^t growth')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Total Population (log scale)')
axes[0].set_title('Population Growth')
axes[0].legend()
axes[0].grid(True)

# Age structure convergence
for i in range(n_classes):
    axes[1].plot(population[:, i] / population.sum(axis=1),
                label=f'Age class {i}')
    axes[1].axhline(stable_dist[i], color=f'C{i}', linestyle='--', alpha=0.4)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Proportion of Population')
axes[1].set_title('Convergence to Stable Age Distribution')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### Fibonacci Sequence as Eigenvalue Problem

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fibonacci recurrence: F(n+1) = F(n) + F(n-1)
# Matrix form: [F(n+1), F(n)]^T = [[1, 1], [1, 0]] @ [F(n), F(n-1)]^T

A_fib = np.array([[1, 1], [1, 0]], dtype=float)

# Eigenvalues
evals = np.linalg.eigvals(A_fib)
lambda1, lambda2 = sorted(evals, reverse=True)
print(f"Eigenvalues: {lambda1:.6f}, {lambda2:.6f}")

# These are the golden ratio and its conjugate
phi = (1 + np.sqrt(5)) / 2
psi = (1 - np.sqrt(5)) / 2
print(f"Golden ratio phi = {phi:.6f}")
print(f"Conjugate psi = {psi:.6f}")

# Binet's formula: F(n) = (phi^n - psi^n) / sqrt(5)
def fibonacci_binet(n):
    return (phi**n - psi**n) / np.sqrt(5)

# Compute Fibonacci numbers
n_terms = 20
fib_binet = np.array([fibonacci_binet(n) for n in range(n_terms)])
print(f"Fibonacci (Binet): {np.round(fib_binet[:10])}")

# Via matrix powers; state = [F(n), F(n-1)], seeded with
# [F(0), F(-1)] = [0, 1] so the indexing matches Binet's formula
fib_matrix = np.zeros(n_terms)
state = np.array([0, 1], dtype=float)
fib_matrix[0] = state[0]
for n in range(1, n_terms):
    state = A_fib @ state
    fib_matrix[n] = state[0]

print(f"Fibonacci (matrix): {np.round(fib_matrix[:10])}")

# Growth rate ratio
ratios = fib_matrix[1:] / fib_matrix[:-1]
print(f"Ratio F(n+1)/F(n) converges to phi = {phi:.6f}")
print(f"Actual ratios: {ratios[-5:]}")

## Spectral Methods for PDEs

### Chebyshev Collocation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eig

# Chebyshev differentiation matrix (the standard construction
# popularized by Trefethen's Spectral Methods in MATLAB)
def chebyshev_diff_matrix(n):
    """Chebyshev differentiation matrix on [-1, 1]"""
    x = np.cos(np.pi * np.arange(n) / (n - 1))
    c = np.ones(n)
    c[0] = 2
    c[-1] = 2
    c = c * (-1)**np.arange(n)

    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                D[i, j] = c[i] / (c[j] * (x[i] - x[j]))

    for i in range(n):
        D[i, i] = -np.sum(D[i, :i]) - np.sum(D[i, i+1:])

    return x, D

# Solve u'' + lambda u = 0 on [-1, 1] with u(-1)=u(1)=0
# This has exact solutions: u_k(x) = sin(k*pi*(x+1)/2), lambda_k = (k*pi/2)^2

n = 40
x, D = chebyshev_diff_matrix(n)
D2 = D @ D

# Apply boundary conditions (u(-1)=u(1)=0)
interior = np.arange(1, n-1)
D2_interior = D2[np.ix_(interior, interior)]

# Solve eigenvalue problem and sort eigenpairs together
evals, evecs = eig(-D2_interior)
idx = np.argsort(evals.real)
evals = evals.real[idx]
evecs = evecs[:, idx].real

# Exact eigenvalues
k = np.arange(1, len(interior)+1)
lambda_exact = (k * np.pi / 2)**2

print("Chebyshev eigenvalues vs exact:")
for i in range(min(5, len(interior))):
    print(f"lambda_{i+1}: {evals[i]:.6f} (exact: {lambda_exact[i]:.6f})")

# Plot eigenfunctions
x_fine = np.linspace(-1, 1, 200)

fig, ax = plt.subplots(figsize=(10, 6))

# Add boundary conditions
x_with_bc = np.concatenate([[x[0]], x[interior], [x[-1]]])

for k in range(1, 4):
    evec = np.zeros(n)
    evec[interior] = evecs[:, k-1]
    # Eigenvectors are defined up to scale and sign: normalize the
    # amplitude to 1 and align the sign with the exact mode
    ref = np.sin(k * np.pi * (x[interior] + 1) / 2)
    evec *= np.sign(evec[interior] @ ref) / np.max(np.abs(evec))
    u_exact_k = np.sin(k * np.pi * (x_fine + 1) / 2)
    ax.plot(x_fine, u_exact_k, label=f'u_{k} (exact)',
           linewidth=2)
    ax.plot(x_with_bc, evec, 'o', markersize=4,
           label=f'u_{k} (Chebyshev)')

ax.set_xlabel('x')
ax.set_ylabel('u(x) (normalized)')
ax.set_title('Eigenfunction Approximation with Chebyshev Collocation')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## Advanced Numerical Methods for Eigenvalue Problems

### Arnoldi and Lanczos Iteration

In [ ]:
import numpy as np
from scipy.sparse import diags
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt

# Create large sparse matrix (1D Laplacian)
n = 1000
diagonals = [-2 * np.ones(n), np.ones(n-1), np.ones(n-1)]
offsets = [0, 1, -1]
A = diags(diagonals, offsets, shape=(n, n), format='csr')

# 10 eigenvalues of largest magnitude (Lanczos), and the 10
# closest to zero (shift-invert Lanczos with sigma = 0)
k = 10
evals_large = eigsh(A, k=k, which='LM', return_eigenvectors=False)
evals_small = eigsh(A, k=k, sigma=0, which='LM',
                    return_eigenvectors=False)

print("10 eigenvalues of largest magnitude:")
print(evals_large)
print("\n10 eigenvalues closest to zero:")
print(evals_small)

# Exact eigenvalues: lambda_j = -4*sin^2(j*pi/(2*(n+1)))
j = np.arange(1, n+1)
lambda_exact = np.sort(-4 * np.sin(j * np.pi / (2 * (n + 1)))**2)

print("\nComparison with exact eigenvalues:")
print(f"Most negative computed: {evals_large.min():.8f}")
print(f"Most negative exact:    {lambda_exact[0]:.8f}")
print(f"Error: {abs(evals_large.min() - lambda_exact[0]):.2e}")

# Convergence study: run m steps of the Lanczos iteration by hand
# and track the error of the extremal Ritz value
def lanczos_extremal(A, m, seed=0):
    """Smallest Ritz value after m Lanczos steps."""
    rng = np.random.default_rng(seed)
    n = A.shape[0]
    Q = np.zeros((n, m + 1))
    alpha = np.zeros(m)
    beta = np.zeros(m + 1)
    q = rng.standard_normal(n)
    Q[:, 0] = q / np.linalg.norm(q)
    for j in range(m):
        w = A @ Q[:, j] - beta[j] * Q[:, j - 1]
        alpha[j] = Q[:, j] @ w
        w -= alpha[j] * Q[:, j]
        # full reorthogonalization (simple but not cheap)
        w -= Q[:, :j+1] @ (Q[:, :j+1].T @ w)
        beta[j + 1] = np.linalg.norm(w)
        Q[:, j + 1] = w / beta[j + 1]
    T = (np.diag(alpha) + np.diag(beta[1:m], 1)
         + np.diag(beta[1:m], -1))
    return np.linalg.eigvalsh(T).min()

ms = np.arange(5, 51, 5)
errors = [abs(lanczos_extremal(A, m) - lambda_exact[0]) for m in ms]

fig, ax = plt.subplots(figsize=(9, 6))
ax.semilogy(ms, errors, 'bo-', linewidth=2, markersize=6)
ax.set_xlabel('Krylov subspace dimension m')
ax.set_ylabel('Error in extremal eigenvalue')
ax.set_title('Lanczos Iteration Convergence')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Large-Scale Eigenvalue Problems with scipy.sparse.linalg

In [ ]:
import numpy as np
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import eigs, LinearOperator
import time

# Create a large random web graph (PageRank-type problem)
np.random.seed(0)
n_nodes = 5000
edges_per_node = 20

def create_random_graph(n, edges_per_node):
    """Create random directed graph as adjacency matrix"""
    G = lil_matrix((n, n))
    for i in range(n):
        j = np.random.choice(n, edges_per_node, replace=False)
        G[i, j] = 1
    return G.tocsr()

G = create_random_graph(n_nodes, edges_per_node)

# Row-stochastic transition matrix P (rows sum to 1)
out_degree = np.asarray(G.sum(axis=1)).flatten()
out_degree[out_degree == 0] = 1
P = G.multiply(1.0 / out_degree[:, np.newaxis]).tocsr()

# Google matrix: A = (1-d)/n * ones + d*P, with damping factor d.
# PageRank is the dominant LEFT eigenvector of A, i.e. the right
# eigenvector of A^T. Forming A densely would destroy sparsity, so
# we provide the matrix-vector product A^T x as a LinearOperator.
d = 0.85
def google_matvec_T(x):
    return d * (P.T @ x) + (1 - d) / n_nodes * x.sum()

A_T = LinearOperator((n_nodes, n_nodes), matvec=google_matvec_T)

t0 = time.time()
evals, evecs = eigs(A_T, k=1, which='LM', v0=np.ones(n_nodes))
t_sparse = time.time() - t0

PageRank = np.abs(evecs[:, 0].real)
PageRank = PageRank / PageRank.sum()

print(f"Sparse eigenvalue computation time: {t_sparse:.4f} seconds")
print(f"Dominant eigenvalue: {evals[0].real:.6f}")
print("Top 10 nodes by PageRank:")
top_nodes = np.argsort(PageRank)[-10:][::-1]
for rank, node in enumerate(top_nodes):
    print(f"  {rank+1}. Node {node}: {PageRank[node]:.6f}")

# Verify eigenvector property
residual = np.linalg.norm(google_matvec_T(PageRank)
                          - evals[0].real * PageRank)
print(f"Residual ||A^T x - lambda*x||: {residual:.2e}")

## Generalized Eigenvalue Problems: Structural and Quantum Applications

### A Two-Mass Vibration Example

In [ ]:
import numpy as np
from scipy.linalg import eigh
import matplotlib.pyplot as plt

# Mass-spring chain: wall--k0--m1--k1--m2--k2--wall
k0, k1, k2 = 1, 2, 1
m1, m2 = 1, 1

# Stiffness matrix
K = np.array([[k0 + k1, -k1],
              [-k1, k1 + k2]], dtype=float)

# Mass matrix
M = np.diag([m1, m2])

# Solve generalized eigenvalue problem
evals_gen, evecs_gen = eigh(K, M)

print("Natural frequencies (rad/s):", np.sqrt(evals_gen))
print("Natural frequencies (Hz):", np.sqrt(evals_gen)/(2*np.pi))
print("\nMode 1 shape:", evecs_gen[:, 0])
print("Mode 2 shape:", evecs_gen[:, 1])

# Verify solution
for i in range(2):
    lam = evals_gen[i]
    v = evecs_gen[:, i]
    residual = np.linalg.norm(K @ v - lam * M @ v)
    print(f"Mode {i+1}: residual = {residual:.2e}")

### Conversion to Standard Form

In [ ]:
import numpy as np
from scipy.linalg import cholesky

# Method: Cholesky transformation for SPD B.
# With B = L L^T and w = L^T v, the problem A v = lambda B v
# becomes the standard symmetric problem C w = lambda w,
# where C = L^{-1} A L^{-T}.

A = np.array([[4, 1], [1, 3]], dtype=float)
B = np.array([[2, 0.5], [0.5, 1]], dtype=float)

# Cholesky decomposition of B
L = cholesky(B, lower=True)
L_inv = np.linalg.inv(L)

# Transformed matrix C = L^{-1} A L^{-T} (symmetric)
C = L_inv @ A @ L_inv.T

# Solve standard problem
evals_c, evecs_c = np.linalg.eigh(C)

# Transform back eigenvectors: v = L^{-T} w
evecs_generalized = L_inv.T @ evecs_c

print("Generalized eigenvalues:", evals_c)

# Verify
for i in range(2):
    lam = evals_c[i]
    v = evecs_generalized[:, i]
    residual = np.linalg.norm(A @ v - lam * B @ v)
    print(f"Mode {i+1}: residual = {residual:.2e}")

## Nonlinear Eigenvalue Problems

### Polynomial Eigenvalue Problems

In [ ]:
import numpy as np

# Second-order system: M*x'' + C*x' + K*x = 0
M = np.eye(2)  # Mass
C = 0.2 * np.eye(2)  # Damping
K = 2 * np.eye(2) - np.eye(2, k=1) - np.eye(2, k=-1)  # Stiffness

n = M.shape[0]
M_inv = np.linalg.inv(M)

# Companion form: [0, I; -M^{-1}K, -M^{-1}C]
A_companion = np.block([[np.zeros((n, n)), np.eye(n)],
                       [-M_inv @ K, -M_inv @ C]])

# Solve
evals, evecs = np.linalg.eig(A_companion)

print(f"Eigenvalues: {evals}")
print(f"Natural frequencies: {np.abs(evals[evals.imag > 0])}")

## Iterative Methods for Large Systems

### Kaczmarz Algorithm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def kaczmarz(A, b, x0, max_iter=100, tol=1e-6):
    """Kaczmarz algorithm for solving Ax = b"""
    m, n = A.shape
    x = x0.copy()
    residuals = []

    for k in range(max_iter):
        i = k % m  # Cycle through rows
        a_i = A[i, :]
        denom = np.dot(a_i, a_i)

        if denom > 1e-15:
            x = x + (b[i] - np.dot(a_i, x)) / denom * a_i

        residual = np.linalg.norm(A @ x - b)
        residuals.append(residual)

        if residual < tol:
            return x, k + 1, residuals

    return x, max_iter, residuals

# Test system (consistent: b is in the range of A)
np.random.seed(0)
m, n = 20, 15
A = np.random.randn(m, n)
x_true = np.random.randn(n)
b = A @ x_true

# Kaczmarz (convergence to 1e-6 takes a few thousand sweeps here)
x_kacz, n_iter, residuals = kaczmarz(A, b, np.zeros(n),
                                     max_iter=5000)

print(f"Kaczmarz converged in {n_iter} iterations")
print(f"Final residual: {residuals[-1]:.2e}")

# Direct solver
x_direct = np.linalg.lstsq(A, b, rcond=None)[0]
print(f"Direct solution error: {np.linalg.norm(x_kacz - x_direct):.2e}")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(residuals, 'b-', linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Residual ||Ax - b||')
ax.set_title('Kaczmarz Convergence')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Conjugate Gradient Least Squares (CGLS)

In [ ]:
import numpy as np

def cgls(A, b, x0=None, max_iter=None, tol=1e-6):
    """Conjugate Gradient Least Squares for Ax = b"""
    m, n = A.shape
    if max_iter is None:
        max_iter = n

    x = x0 if x0 is not None else np.zeros(n)
    r = b - A @ x
    p = A.T @ r
    gamma = np.dot(p, p)
    residuals = []

    for k in range(max_iter):
        Ap = A @ p
        alpha = gamma / np.dot(Ap, Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        s = A.T @ r
        gamma_new = np.dot(s, s)
        residuals.append(np.linalg.norm(A @ x - b))

        if np.sqrt(gamma_new) < tol:
            return x, k + 1, residuals

        beta = gamma_new / gamma
        p = s + beta * p
        gamma = gamma_new

    return x, max_iter, residuals

# Overdetermined system
np.random.seed(0)
m, n = 100, 30
A = np.random.randn(m, n)
x_true = np.random.randn(n)
b = A @ x_true + 0.01 * np.random.randn(m)

# CGLS
x_cgls, n_iter, residuals = cgls(A, b)
print(f"CGLS converged in {n_iter} iterations")
print(f"Final residual: {residuals[-1]:.2e}")

# Comparison
x_normal = np.linalg.lstsq(A, b, rcond=None)[0]
print(f"Solution difference: {np.linalg.norm(x_cgls - x_normal):.2e}")